<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/12_first_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
compounds = pd.read_csv(f'{BASE}/data/processed/compounds.csv', index_col='name')
sigpca = pd.read_parquet(f'{BASE}/data/processed/signature_pca.parquet')
fp = pd.read_parquet(f'{BASE}/data/processed/ecfp4.parquet')
matrix = pd.read_parquet(f'{BASE}/results/hallmark_matrix_baseline.parquet')
print(len(compounds), sigpca.shape, fp.shape)

Mounted at /content/drive
15542 (15542, 100) (15542, 2048)


In [2]:
!wget -nc -P {BASE}/data/raw https://s3.amazonaws.com/data.clue.io/repurposing/downloads/repurposing_drugs_20200324.txt
hub = pd.read_csv(f'{BASE}/data/raw/repurposing_drugs_20200324.txt', sep='\t', comment='!')
print(hub.columns.tolist())

hub['name'] = hub['pert_iname'].str.lower()
hub = hub.drop_duplicates('name').set_index('name')
hub = hub[hub.index.isin(compounds.index)]
targets = hub['target'].dropna().str.split('|')
print(len(targets), 'VCAP compounds have target information')
print('sirolimus targets:', targets.get('sirolimus'))

counts = targets.explode().value_counts()
keep = set(counts[counts >= 5].index)
T = pd.DataFrame(0, index=compounds.index, columns=[f'tg_{t}' for t in sorted(keep)], dtype=np.uint8)
for name, ts in targets.items():
    T.loc[name, [f'tg_{t}' for t in ts if t in keep]] = 1
T['has_targets'] = compounds.index.isin(targets.index).astype(np.uint8)
T.to_parquet(f'{BASE}/data/processed/targets.parquet')
print(T.shape)

--2026-10-13 20:04:26--  https://s3.amazonaws.com/data.clue.io/repurposing/downloads/repurposing_drugs_20200324.txt
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.182.17.40, 16.15.212.241, 16.15.255.219, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.182.17.40|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 530302 (518K) [text/plain]
Saving to: ‘/content/drive/MyDrive/longevity_project/data/raw/repurposing_drugs_20200324.txt’

repurposing_drugs_2 100%[===================>] 517.87K  --.-KB/s    in 0.07s   

2026-10-13 20:04:26 (6.79 MB/s) - ‘/content/drive/MyDrive/longevity_project/data/raw/repurposing_drugs_20200324.txt’ saved [530302/530302]

['pert_iname', 'clinical_phase', 'moa', 'target', 'disease_area', 'indication']
1509 VCAP compounds have target information
sirolimus targets: ['FKBP1A', 'MTOR']
(15542, 268)


In [3]:
def similarity(a, b):
    if not (compounds.loc[a, 'has_structure'] and compounds.loc[b, 'has_structure']):
        return 0.0
    x, y = fp.loc[a].values.astype(bool), fp.loc[b].values.astype(bool)
    return (x & y).sum() / (x | y).sum()

def chem_groups(drugs, cutoff=0.6):
    groups = []
    for d in drugs:
        for g in groups:
            if any(similarity(d, m) >= cutoff for m in g):
                g.append(d)
                break
        else:
            groups.append([d])
    return groups

print(chem_groups([d for d in kd['by_hallmark']['nutrient_sensing']['helpful'] if d in compounds.index]))

[['sirolimus', 'everolimus', 'temsirolimus'], ['pp242'], ['ku-0063794'], ['metformin'], ['resveratrol'], ['curcumin']]


In [4]:
rng = np.random.default_rng(42)
holdout = {}
for hm, k in kd['by_hallmark'].items():
    helpful = [d for d in k['helpful'] if d in compounds.index]
    target_n = len(helpful) // 4
    held = []
    for i in rng.permutation(len(groups := chem_groups(helpful))):
        if len(held) + len(groups[i]) <= target_n:
            held += groups[i]
    holdout[hm] = held
    print(f'{hm:17} {len(helpful):2} helpful drugs | held out: {", ".join(held) or "none"}')

json.dump(holdout, open(f'{BASE}/data/processed/holdout.json', 'w'), indent=2)

senescence         6 helpful drugs | held out: apigenin
nutrient_sensing   8 helpful drugs | held out: ku-0063794, resveratrol
mitochondria       4 helpful drugs | held out: rosiglitazone
autophagy          6 helpful drugs | held out: pp242
inflammation      12 helpful drugs | held out: parthenolide, curcumin, aspirin
proteostasis       3 helpful drugs | held out: none


In [5]:
all_known = {d for k in kd['by_hallmark'].values() for d in k['helpful'] + k['harmful']}
pool = [n for n in compounds.index[compounds['is_named']] if n not in all_known]
background = list(np.random.default_rng(0).choice(pool, 300, replace=False))
json.dump(background, open(f'{BASE}/data/processed/background_negatives.json', 'w'))

def training_set(hm):
    k = kd['by_hallmark'][hm]
    pos = [d for d in k['helpful'] if d in compounds.index and d not in holdout[hm]]
    neg = [d for d in k['harmful'] if d in compounds.index] + background
    return pos, neg

for hm in kd['by_hallmark']:
    pos, neg = training_set(hm)
    print(f'{hm:17} positives: {len(pos):2} | negatives: {len(neg)}')

senescence        positives:  5 | negatives: 306
nutrient_sensing  positives:  6 | negatives: 300
mitochondria      positives:  3 | negatives: 302
autophagy         positives:  5 | negatives: 300
inflammation      positives:  9 | negatives: 304
proteostasis      positives:  3 | negatives: 309


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = sigpca

def fit(pos, neg):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000))
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    return model.fit(X.loc[pos + neg].values, y)

rows = []
for hm in kd['by_hallmark']:
    pos, neg = training_set(hm)
    groups = chem_groups(pos)
    if len(groups) < 2:
        print(f'{hm}: too few chemical groups to test, skipped')
        continue
    for g in groups:
        train_pos = [d for d in pos if d not in g]
        model = fit(train_pos, neg)
        rest = X.index.difference(train_pos + neg)
        pct = pd.Series(model.predict_proba(X.loc[rest].values)[:, 1], index=rest).rank(pct=True) * 100
        for d in g:
            rows.append({'hallmark': hm, 'drug': d, 'ml_pct': pct[d],
                         'baseline_pct': matrix.loc[d, f'{hm}_pct']})

res = pd.DataFrame(rows).round(1)
summary = res.groupby('hallmark')[['ml_pct', 'baseline_pct']].mean().round(1)
summary['drugs_tested'] = res.groupby('hallmark').size()
print(summary)
res.to_csv(f'{BASE}/results/ml_v1_cv.csv', index=False)

                  ml_pct  baseline_pct  drugs_tested
hallmark                                            
autophagy           56.0          50.3             5
inflammation        47.6          59.1             9
mitochondria        46.7          72.3             3
nutrient_sensing    87.9          70.8             6
proteostasis        75.5          65.6             3
senescence          40.5          40.0             5


In [7]:
ml = pd.DataFrame(index=compounds.index)
for hm in kd['by_hallmark']:
    pos, neg = training_set(hm)
    ml[hm] = fit(pos, neg).predict_proba(X.loc[compounds.index].values)[:, 1]

ml_pct = (ml.rank(pct=True) * 100).round(1)
out = ml.round(4).join(ml_pct.add_suffix('_pct'))
out.to_parquet(f'{BASE}/results/ml_scores_v1.parquet')

for hm in kd['by_hallmark']:
    pos, neg = training_set(hm)
    top = ml_pct.loc[compounds['is_named'] & ~ml_pct.index.isin(pos + neg), hm].nlargest(8)
    print(f'{hm:17}', ', '.join(top.index))

senescence        alvespimycin, epitestosterone, md-041, geldanamycin, ku-c103672, anisomycin, prednisone, icariin
nutrient_sensing  alvespimycin, epitestosterone, mitomycin-c, amsacrine, importazole, withaferin-a, puromycin, sulconazole
mitochondria      penfluridol, irinotecan, scriptaid, pyrimethamine, cytarabine, sa-1973507, sn-38, le-135
autophagy         alvespimycin, amsacrine, cytarabine, md-041, withaferin-a, irinotecan, epitestosterone, mycophenolate-mofetil
inflammation      icariin, lumicolchicine, alvespimycin, gw-1929, anisomycin, amsacrine, abiraterone, 4,5,6,7-tetrabromobenzotriazole
proteostasis      alvespimycin, withaferin-a, irinotecan, mycophenolate-mofetil, amsacrine, cytarabine, sn-38, epitestosterone


In [8]:
s = summary.reset_index()
parts = ', '.join(f"{r.hallmark.replace('_', ' ')} {r.ml_pct:.0f} vs {r.baseline_pct:.0f}" for r in s.itertuples())
print(f"Trained regularized logistic regression per hallmark on signature PCA features "
      f"(positives: known helpful drugs; negatives: harmful drugs plus 300 random compounds). "
      f"Grouped cross-validation, average percentile of left-out drugs, ML vs baseline: {parts}. "
      f"Held out {sum(len(v) for v in holdout.values())} known drugs for Week 6; added target features "
      f"for {int(T['has_targets'].sum())} compounds.")

Trained regularized logistic regression per hallmark on signature PCA features (positives: known helpful drugs; negatives: harmful drugs plus 300 random compounds). Grouped cross-validation, average percentile of left-out drugs, ML vs baseline: autophagy 56 vs 50, inflammation 48 vs 59, mitochondria 47 vs 72, nutrient sensing 88 vs 71, proteostasis 76 vs 66, senescence 40 vs 40. Held out 8 known drugs for Week 6; added target features for 1509 compounds.
